In [1]:
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    !pip install --no-deps unsloth vllm==0.8.5.post1

In [2]:
#@title Colab Extra Install { display-mode: "form" }
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    !pip install --no-deps unsloth vllm==0.8.5.post1
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    # Skip restarting message in Colab
    import sys, re, requests; modules = list(sys.modules.keys())
    for x in modules: sys.modules.pop(x) if "PIL" in x or "google" in x else None
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer

    # vLLM requirements - vLLM breaks Colab due to reinstalling numpy
    f = requests.get("https://raw.githubusercontent.com/vllm-project/vllm/refs/heads/main/requirements/common.txt").content
    with open("vllm_requirements.txt", "wb") as file:
        file.write(re.sub(rb"(transformers|numpy|xformers)[^\n]{1,}\n", b"", f))
    !pip install -r vllm_requirements.txt

Ignoring importlib_metadata: markers 'python_version < "3.10"' don't match your environment
Ignoring six: markers 'python_version > "3.11"' don't match your environment
Ignoring setuptools: markers 'python_version > "3.11"' don't match your environment


In [3]:
from unsloth import FastModel
import torch
max_seq_length = 512
# các model sử dụng trong quá trình
fourbit_models = [
    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-1b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-27b-it-unsloth-bnb-4bit",

    # Other popular models!
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/Llama-3.3-70B",
    "unsloth/mistral-7b-instruct-v0.3",
    "unsloth/Phi-4",
] # More models at https://huggingface.co/unsloth

# Load model và model cho việc token
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B",
    max_seq_length = max_seq_length, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)
tokenizer.chat_template ="""
{% for message in messages %}
{% if message['role'] == 'system' %}
<|start_of_turn|><|system|>
{{ message['content'] }}<|end_of_turn|>
{% elif message['role'] == 'user' %}
<|start_of_turn|><|user|>
{{ message['content'] }}<|end_of_turn|>
{% elif message['role'] == 'assistant' %}
<|start_of_turn|><|assistant|>
{{ message['content'] }}<|end_of_turn|>
{% endif %}
{% endfor %}
""".strip()
print("200 OK")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-06-25 06:17:33.342934: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750832253.365986     881 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750832253.373320     881 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 06-25 06:17:40 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 06-25 06:17:40 [__init__.py:239] Automatically detected platform cuda.
==((====))==  Unsloth 2025.6.5: Fast Llama patching. Transformers: 4.51.3. vLLM: 0.8.5.post1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
200 OK


In [4]:
# Thư viện cho LLM
from transformers import (
    AutoModelForCausalLM, # Tinh chỉnh mô hình trả lời nhân quả
    AutoTokenizer, # Token hóa câu văn
    BitsAndBytesConfig, # điều chỉnh hiệu quả đông bộ nhớ
    HfArgumentParser,
    TrainingArguments, # Quản lý tham số,
    pipeline,
    logging # ghi lại thông tin loss, accuracy
)

# Tinh chỉnh adapter
from peft import (
    LoraConfig, # Cấu hình cho lora
    PeftModel, # Lưu trữ pp fine tune hiệu quả
    prepare_model_for_kbit_training, # model để training
    get_peft_model # lấy ra sau khi fineTune
)

import os, torch, wandb # wandb giúp lưu trư các thông tin trong quá trình train và vẽ biểu đồ
from datasets import load_dataset
from trl import SFTTrainer, setup_chat_format # Thực hiện training bằng supervised
import bitsandbytes as bnb
print("200 OK")

200 OK


In [ ]:
from google.colab import userdata
import os
wandb_token = os.getenv("WANDB_API_KEY") or userdata.get("WANDB_API_KEY")
if not wandb_token:
    raise ValueError("Missing WANDB_API_KEY. Set env var or add to Colab secrets.")
wandb.login(key=wandb_token)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: onichanht1234 (onichanht1234-i-h-c-qu-c-gia-tp-hcm) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [6]:
from typing import List
lora_target_modules: List[str] = [
        "q_proj",
        "v_proj",
]
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # SHould leave on always!

    r = 8,           # Larger = higher accuracy, but might overfit
    lora_alpha = 8,  # Recommended alpha == r at least
    lora_dropout = 0.05,
    bias = "none",
    target_modules= lora_target_modules,
    random_state = 3407,
    task_type="CAUSAL_LM"
)

Unsloth: Making `model.base_model.model.model` require gradients


In [7]:
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer
from huggingface_hub import login, create_repo# thư viện để login dựa vào api key
# Load tập train
dataset = load_dataset("Thang21/chatdocter", split="train")
print(dataset)
print(f"Số lượng hàng của dataset sau khi load: {dataset.num_rows}")

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 177627
})
Số lượng hàng của dataset sau khi load: 177627


In [8]:
# Test coi chia có đúng chưa
dataNoTrain = dataset.train_test_split(test_size = 0.2, seed = 42)
datasetNotrain= dataNoTrain['test']
datasetNotrain_train = dataNoTrain['train']
print(datasetNotrain_train.num_rows)

142101


In [9]:
datasets = [dataset.shard(num_shards=20, index=i) for i in range(20)]
test_data = datasets[0]

train_val_dict = test_data.train_test_split(test_size=0.2, seed = 42)
# Tập train
train_data = train_val_dict['train']
# Tập val
cross_data = train_val_dict['test']
print(f"train length {train_data.num_rows}")
print(f"cross length {cross_data.num_rows}")

train length 7105
cross length 1777


In [10]:
### có batch

system_prompt = """Bạn là một trợ lý ảo và tư vấn viên chuyên nghiệp cho người Việt Nam.
Nhiệm vụ của bạn là trả lời tất cả câu hỏi liên quan đến kiến thức chung (khoa học, xã hội, đời sống, giáo dục, v.v.) bằng tiếng Việt có thể dùng thuật ngữ tiếng Anh.
Bạn cần sử dụng ngôn ngữ dễ hiểu, văn phong lịch sự, gần gũi, và luôn thể hiện sự tận tình, chu đáo trong từng câu trả lời."""

def format_data_for_sft(example, tokenizer, system_prompt):
    """
    Định dạng một ví dụ dữ liệu thành một chuỗi văn bản duy nhất
    theo chat template của tokenizer.

    Hàm này sẽ được dùng trong dataset.map()
    """
    messages = []

    # 1. Thêm tin nhắn của người dùng
    user_message_content = ""
    if example.get('input') and example['input'].strip():
        user_message_content = f"{system_prompt}\n\nCâu hỏi: {example['instruction']}\nThông tin thêm: {example['input']}"
    else:
        user_message_content = f"{system_prompt}\n\nCâu hỏi: {example['instruction']}"

    messages.append({"role": "user", "content": user_message_content})

    # 2. Thêm câu trả lời của trợ lý ảo (model)
    messages.append({"role": "assistant", "content": example['output']})

    # 3. Sử dụng hàm apply_chat_template của tokenizer để tạo chuỗi văn bản cuối cùng.
    #    Hàm này sẽ tự động thêm các token đặc biệt như <start_of_turn>, v.v.
    #    tokenize=False: chỉ trả về string, không trả về input_ids.
    #    add_generation_prompt=False: không thêm prompt để model sinh tiếp, vì ta có cả câu trả lời.
    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    # 4. Trả về một dictionary với một cột duy nhất là "text"
    return { "text": formatted_text }

formatted_train_data = train_data.map(
    lambda example: format_data_for_sft(example, tokenizer, system_prompt),
    num_proc=4 # Sử dụng nhiều tiến trình để tăng tốc
)

formatted_cross_data = cross_data.map(
    lambda example: format_data_for_sft(example, tokenizer, system_prompt),
    num_proc=4
)

# In ra một ví dụ để kiểm tra
print("Ví dụ dữ liệu sau khi định dạng:")
print(formatted_train_data[0]['text'])


Ví dụ dữ liệu sau khi định dạng:
<|start_of_turn|><|user|>
Bạn là một trợ lý ảo và tư vấn viên chuyên nghiệp cho người Việt Nam.
Nhiệm vụ của bạn là trả lời tất cả câu hỏi liên quan đến kiến thức chung (khoa học, xã hội, đời sống, giáo dục, v.v.) bằng tiếng Việt có thể dùng thuật ngữ tiếng Anh.
Bạn cần sử dụng ngôn ngữ dễ hiểu, văn phong lịch sự, gần gũi, và luôn thể hiện sự tận tình, chu đáo trong từng câu trả lời.

Câu hỏi: Nếu bạn là bác sĩ, vui lòng trả lời các câu hỏi y khoa dựa trên mô tả của bệnh nhân.
Thông tin thêm: vi: Xin chào, tôi đến từ new delhi, 26 tuổi. tôi đang gặp một vấn đề về cương cứng. dương vật của tôi không cứng lắm và đôi khi tôi phóng tinh mà không cương đúng cách. tinh hoàn và Dương vật tôi gần như bị thắt chặt (giống như nó bị thu nhỏ lại).<|end_of_turn|>
<|start_of_turn|><|assistant|>
vi: Xin chào bạn thân mến, đầu tiên, đừng để bị ảnh hưởng bởi bất kỳ loại thuốc không kê đơn nào tuyên bố tăng kích thước dương vật hoặc cải thiện hiệu suất...nó có thể gây hạ

In [11]:
output_model = 'PhongGoldFish/testing_finetune'
from transformers import TrainingArguments
from trl import SFTTrainer
from huggingface_hub import login # thư viện để login dựa vào api key
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("Qlora")

# login vào huggingface
login(token = secret_value_0)

# Đây là cấu hình TrainingArguments đã được tối ưu cho Unsloth
training_arguments = TrainingArguments(
    output_dir=output_model,
    per_device_train_batch_size = 2, # Thử tăng lên 4 với Unsloth
    per_device_eval_batch_size = 4,  # Đồng bộ
    gradient_accumulation_steps = 2, # Giảm xuống để effective batch size = 16

    optim = "paged_adamw_8bit", # Unsloth khuyến nghị 8bit
    num_train_epochs = 2,

    # Sử dụng warmup_ratio thay cho steps để linh hoạt hơn
    warmup_ratio = 0.1,
    weight_decay=0.01,

    # --- PHẦN TỰ ĐỘNG CHỌN ĐỊNH DẠNG ---
    # Kiểm tra xem GPU có hỗ trợ bfloat16 không
    bf16 = torch.cuda.is_bf16_supported(),
    # Nếu không hỗ trợ bf16, thì sử dụng fp16
    fp16 = not torch.cuda.is_bf16_supported(),
    # ------------------------------------

    logging_steps = 10,
    eval_strategy = "steps",
    eval_steps = 100, # Đánh giá thường xuyên hơn để kiểm tra

    learning_rate = 3e-4,
    lr_scheduler_type = "cosine",

    report_to="wandb",
)


trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_train_data,
    eval_dataset = formatted_cross_data,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    num_train_epochs = 3,
    args = training_arguments
)

# Bây giờ bạn có thể bắt đầu training
trainer.train()
wandb.finish()

local_dir = "/kaggle/working/saved_model"
trainer.model.save_pretrained(local_dir)
trainer.tokenizer.save_pretrained(local_dir)

create_repo(output_model,exist_ok = True)
trainer.model.push_to_hub(repo_id = output_model, repo_path_or_name=local_dir)
trainer.tokenizer.push_to_hub(repo_id = output_model, repo_path_or_name=local_dir)
print("200 OK")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,105 | Num Epochs = 2 | Total steps = 1,776
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 2,293,760/3,000,000,000 (0.08% trained)
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss
100,1.580200,1.537938
200,1.468400,1.484016
300,1.514700,1.459122
400,1.514200,1.439447
500,1.476100,1.426541
600,1.474400,1.414279
700,1.371600,1.404479
800,1.423200,1.396190
900,1.364500,1.388119
1000,1.355000,1.382376


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


eval/loss,█▆▅▄▄▃▃▂▂▂▂▁▁▁▁▁▁
eval/runtime,█▁▃▄▃▄▄▆▃▅▂▂▁▂▃▄▃
eval/samples_per_second,▁█▆▅▆▅▅▄▆▄▇▇█▇▆▅▆
eval/steps_per_second,▁█▇▅▇▅▅▄▇▄▇▇█▇▇▅▇
train/epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▇▇▇█████
train/global_step,▁▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/grad_norm,▁█▅▄▃▂▂▃▃▂▃▃▃▃▃▄▃▃▃▃▄▄▄▄▄▄▄▄▃▄▄▄▄▄▄▃▄▄▄▄
train/learning_rate,▁▂▄▄▅███████▇▇▇▇▆▆▆▆▅▅▅▄▄▃▃▃▃▂▂▂▁▁▁▁▁▁▁▁
train/loss,█▇▃▃▂▂▂▂▃▂▂▂▂▂▂▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂
eval/loss,1.36446
eval/runtime,486.1659


...

Saved model to https://huggingface.co/PhongGoldFish/testing_finetune


README.md:   0%|          | 0.00/47.0 [00:00<?, ?B/s]

  0%|          | 0/1 [00:00<?, ?it/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

200 OK
